In [16]:
from sklearn.feature_extraction.text import CountVectorizer
import glob
import pandas as pd
import nltk
from nltk import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag

In [17]:
# modeli za ustrezno obdelavo stavkov, besed, ločil....
nltk.download('punkt')     # stavki, besede
nltk.download('wordnet') #lemmatizacija
nltk.download('averaged_perceptron_tagger') #POS tagganje
nltk.download('omw-1.4') 
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_percep

True

In [18]:
# tokenization and lemmatization 
lemmatizer= WordNetLemmatizer()

In [19]:
# pokupčkamo besede s podobnim korenom, pomenom skupaj
# run, runs, running -> run
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('RB'):
        return wordnet.ADV
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    else:
        return wordnet.NOUN

In [20]:
def tokenize_lematize(tekst):
    tokens = word_tokenize(tekst.lower())  # vse besede pišemo z malo začetnico
    tagged = pos_tag(tokens)
    return [
        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in tagged
        if word.isalpha()    # znebimo se ločil, st,...
    ]

In [33]:
base_vectorizer = CountVectorizer(stop_words='english')
base_stopwords = base_vectorizer.get_stop_words()

custom_words = {
    'book', 'novel', 'story', 'reader',
    'edition', 'classic', 'introduction',
    'publish', 'note', 'cover', 'series',
    'time', 'year', 'new', 'make', 'tell',
    'begin', 'just', 'work', 'face',
    'place', 'mean', 'text', 'author', 'original', 'u', 'seller',
    'masterpiece', 'literature', 'best', 'read', 'man', 'men', 'woman', 'life',
    'fiction', 'tale', 'want', 'come', 'know'
}

all_stopwords = list(base_stopwords.union(custom_words))

In [34]:
# CountVectorizer odstrani 'stopwords' in ustvari nenegativno matriko, na (i, j)-tem mestu
# imamo pojavitev besede i v j-tem dokumentu (glej zapiske na tablici)


# vzamemo 49/50 knjig, eno bomo potem poskusali uvrstiti med žanre
filepaths = glob.glob(r'C:\Users\mokro\Desktop\diploma\najbolj_popularne\najboljse_knjige_opisi\*.txt')[:250]
# min_df=2, max_df=0.9 odstranita redke in pogoste besede, to uniči celoten rezultat
vectorizer= CountVectorizer(stop_words=all_stopwords, 
                            tokenizer= tokenize_lematize,
                            input = 'filename', 
                            encoding='latin-1', 
                            min_df=1, 
                            max_df=0.9)

In [35]:
X = vectorizer.fit_transform(filepaths) 


c:\Users\mokro\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\mokro\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['far'] not in stop_words.
  warnings.warn(


In [36]:
print(X)

# malo lepše, prikaz
dense_matrix = X.toarray()
print(dense_matrix)

#še lepše
feature_names = vectorizer.get_feature_names_out()
df = pd.DataFrame(dense_matrix, columns=feature_names)
print(df.head())

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 13575 stored elements and shape (250, 5544)>
  Coords	Values
  (0, 5445)	2
  (0, 1763)	1
  (0, 1923)	1
  (0, 2925)	1
  (0, 751)	1
  (0, 1185)	3
  (0, 2367)	2
  (0, 1999)	3
  (0, 4222)	1
  (0, 3373)	1
  (0, 149)	1
  (0, 2860)	1
  (0, 3300)	1
  (0, 3515)	1
  (0, 4444)	1
  (0, 684)	2
  (0, 4795)	1
  (0, 3476)	1
  (0, 1368)	2
  (0, 2201)	1
  (0, 1098)	1
  (0, 2877)	1
  (0, 1902)	1
  (0, 4378)	1
  (0, 553)	1
  :	:
  (249, 3174)	1
  (249, 1345)	1
  (249, 199)	1
  (249, 3552)	1
  (249, 1766)	1
  (249, 4330)	1
  (249, 1335)	1
  (249, 976)	1
  (249, 5089)	1
  (249, 2824)	1
  (249, 513)	1
  (249, 4566)	1
  (249, 1336)	1
  (249, 3274)	1
  (249, 4450)	1
  (249, 95)	1
  (249, 1018)	1
  (249, 1519)	1
  (249, 791)	1
  (249, 4875)	1
  (249, 1494)	1
  (249, 2879)	1
  (249, 837)	2
  (249, 2691)	2
  (249, 2759)	1
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
   aaron  abag

In [37]:
from sklearn.decomposition import PCA

In [41]:
pca = PCA(n_components=5)  
X_pca = pca.fit_transform(X)

In [42]:
print(pca.explained_variance_ratio_)
print("Skupaj:", sum(pca.explained_variance_ratio_))

[0.01936105 0.01924698 0.01605427 0.0147376  0.01416965]
Skupaj: 0.08356954807823577


In [43]:
feature_names = vectorizer.get_feature_names_out()

for i, comp in enumerate(pca.components_):
    top_words = [feature_names[j] for j in comp.argsort()[-15:]]
    print(f"PC{i+1}: {top_words}")

PC1: ['dead', 'secret', 'room', 'guest', 'seven', 'rhyme', 'devon', 'chop', 'hang', 'sit', 'leave', 'murder', 'love', 'boy', 'little']
PC2: ['earth', 'blood', 'hathaway', 'dead', 'dragomir', 'academy', 'protect', 'vampire', 'friend', 'world', 'dimitri', 'strigoi', 'lissa', 'love', 'rise']
PC3: ['seal', 'student', 'dark', 'lissa', 'friend', 'wizard', 'school', 'lord', 'boy', 'rise', 'potter', 'little', 'voldemort', 'hogwarts', 'harry']
PC4: ['mark', 'page', 'humor', 'brooklyn', 'compassion', 'bad', 'father', 'mccourt', 'irish', 'mother', 'bear', 'miserable', 'harry', 'childhood', 'frank']
PC5: ['voldemort', 'magister', 'girl', 'jem', 'sister', 'heart', 'hogwarts', 'dark', 'infernal', 'device', 'shadowhunters', 'secret', 'harry', 'tessa', 'love']


In [29]:
# import matplotlib.pyplot as plt
# labels = X_pca.argmax(axis=1)   

# pairs = [(0,1), (0,2), (0,3), (1,2), (1,3), (2,3)]

# for i, j in pairs:
#     plt.figure()
#     plt.scatter(
#         X_pca[:, i],
#         X_pca[:, j],
#         c=labels,       
#         cmap="tab10"    
#     )
#     plt.xlabel(f"PC{i+1}")
#     plt.ylabel(f"PC{j+1}")
#     plt.title(f"PC{i+1} vs PC{j+1}")
#     plt.colorbar(label="žanr")   
#     plt.show()

# plt.show()